In [1]:
import pandas as pd
from pathlib import Path
import sys

sys.path.append("../..")  # adjust to wherever data_utils.py sits relative to this notebook
from data_utils import DATASETS

SPLITS_DIR = Path("../../../Data/Data_Collection/Final/Stage_5_Model_Ready/04_splits")

for dataset in ["agg_means", "agg_full_moments", "panel"]:
    path = SPLITS_DIR / "Split_D" / f"{dataset}_train.parquet"
    df = pd.read_parquet(path)
    all_cols = set(df.columns)
    meta_cols = DATASETS[dataset]["meta"]

    missing_meta = meta_cols - all_cols   # expected meta col not actually in file
    feature_cols = all_cols - meta_cols

    print(f"\n{dataset}")
    print(f"  total columns in file:  {len(all_cols)}")
    print(f"  meta columns expected:  {sorted(meta_cols)}")
    print(f"  meta columns MISSING from file: {sorted(missing_meta) or 'none'}")
    print(f"  -> feature columns used: {len(feature_cols)}")

    # Explicitly hunt for anything that LOOKS like a target/date/id but
    # isn't in meta_cols -- this is the actual leakage check, not just a count.
    suspects = [c for c in feature_cols if any(
        kw in c.lower() for kw in
        ["date", "target", "minret", "y_binary", "permno", "dlyret", "dlycap"]
    )]
    print(f"  SUSPICIOUS feature names (would be leakage if genuinely target-like): "
          f"{suspects or 'none'}")


agg_means
  total columns in file:  578
  meta columns expected:  ['date', 'minret_5d_pct', 'target_daily_return', 'y_binary']
  meta columns MISSING from file: none
  -> feature columns used: 574
  SUSPICIOUS feature names (would be leakage if genuinely target-like): ['dlyretx']

agg_full_moments
  total columns in file:  1703
  meta columns expected:  ['date', 'minret_5d_pct', 'target_daily_return', 'y_binary']
  meta columns MISSING from file: none
  -> feature columns used: 1699
  SUSPICIOUS feature names (would be leakage if genuinely target-like): ['dlyretx_cwstd', 'dlyretx_cwskew', 'dlyretx_cwmean', 'dlyretx_spread', 'dlyretx_cwkurt']

panel
  total columns in file:  580
  meta columns expected:  ['date', 'dlycap', 'dlyret', 'minret_5d_pct', 'permno', 'y_binary']
  meta columns MISSING from file: none
  -> feature columns used: 574
  SUSPICIOUS feature names (would be leakage if genuinely target-like): ['dlyretx']


In [2]:
# Confirm minret_5d_pct at row t does NOT include day t's own return.
# Uses the panel table specifically, since panel's dlyret is at native
# per-day granularity (unlike the aggregate's forward-shifted target_daily_return).
sample = pd.read_parquet(SPLITS_DIR / "Split_D" / "panel_train.parquet",
                         columns=["permno", "date", "dlyret", "minret_5d_pct"])
sample = sample.sort_values(["permno", "date"]).reset_index(drop=True)

# For one stock, manually recompute minret_5d_pct at a few rows and compare.
one = sample[sample["permno"] == sample["permno"].iloc[0]].reset_index(drop=True)
for i in [10, 100, 500]:
    fwd = one["dlyret"].iloc[i+1:i+6].values  # t+1 .. t+5, EXCLUDING day i itself
    expected = 100 * fwd.min()
    actual = one["minret_5d_pct"].iloc[i]
    print(f"row {i}: expected={expected:.4f}  actual={actual:.4f}  "
          f"match={abs(expected-actual) < 1e-6}")

row 10: expected=-1.2403  actual=-1.2403  match=True
row 100: expected=-1.6979  actual=-1.6979  match=True
row 500: expected=-1.0342  actual=-1.0342  match=True


In [2]:
# %% [markdown]
# # 05 — Dense MLP (3-Seed Reproducibility)
#
# Standard MLP baseline with SiLU activation.
# Layer widths are DERIVED PER DATASET from the taxonomy, not fixed:
#   [n_features, n_subthemes, n_themes, 1]
#
# EDIT (post Stage_5_Themes/04_numbering_subthemes.ipynb): the old pipeline
# used a single N_SUBTHEMES=107, N_THEMES=13 pair for BOTH feature sets,
# because the old taxonomy happened to give means-only and full-moments the
# same subtheme count. That is no longer true. agg_full_moments carries
# derived "Std"/"Spread"/"Shape" subthemes (from cwstd/spread/cwskew+cwkurt)
# that have no means-only equivalent, so the two datasets now have GENUINELY
# DIFFERENT subtheme counts:
#     agg_means         574 features   128 subthemes   13 themes
#     agg_full_moments 1699 features   331 subthemes   13 themes
# (exact counts read from the taxonomy at runtime below, not hardcoded --
# see ARCHITECTURE section). A single shared width pair is structurally
# wrong for at least one of the two datasets no matter what value is
# chosen, so widths are now looked up per dataset via
# data_utils.load_theme_assignment(), matching exactly what
# SparseKAN.from_taxonomy()/SparseMLP.from_taxonomy() derive for the
# sparse models -- keeping the dense-vs-sparse comparison at matched
# topology.
#
# SiLU chosen deliberately: the KAN's base component is SiLU(x) * base_weight.
# With all spline coefficients zero, the KAN reduces to this MLP.
# Any performance difference is purely attributable to the B-splines.
#
# Fixed architecture per dataset:
#   - Layers: [n_features, n_subthemes, n_themes, 1]  (see ARCHITECTURE)
#   - Activation: SiLU (matching KAN's base activation)
#   - No dropout, no batch norm, no residual connections
#     (KAN has none of these — fair comparison)
#
# 3 seeds × 4 splits × 2 datasets × 2 targets = 48 runs
# Each seed runs the FULL pipeline independently:
#   - Optuna (40 trials) with seeded TPE sampler
#   - Final retrain with best params
#   - Evaluation on train/val/test
#
# This measures true end-to-end reproducibility: if Optuna picks
# different hyperparameters across seeds, that tells us the search
# landscape is flat or noisy. If metrics vary, that tells us the
# model is sensitive to initialisation. Both are important findings.
#
# Regularisation: L1 on weight matrices (analogous to KAN edge-pruning L1)
#
# Target notes:
#   binary:     y_binary (minret_5d_pct < -2.0), BCEWithLogitsLoss, early stop on AUC
#   continuous: minret_5d_pct (raw percentage, NOT a z-score -- see data_utils.py),
#               HuberLoss(delta=<per-split value from huber_delta.json>),
#               early stop on R²
#   Both use Optuna direction="maximize" (AUC and R² are both higher-is-better).
#
# Results save to Drive per-seed as they go — safe against disconnection.
# Aggregated cross-seed summary saved at the end.
# Backtest results (Sharpe/Sortino/cost-sensitivity/etc.) are also saved to
# disk this run, not just printed -- see the final section.
#
# Estimated runtime on T4: ~3-6 hours (3 × ~1-2 hours — MLP is faster than KAN)
# Runtime disconnects automatically when finished.

# %%
# ── COLAB SETUP ──
!pip install -q optuna

from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import numpy as np
import pandas as pd
import random
import time
import torch
import torch.nn as nn
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import (
    evaluate_model, save_predictions, compute_calibration,
    load_predictions, run_full_backtest,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

# EDIT: fresh results dir -- do not point this at any pre-existing
# dense_mlp/ folder from the old pipeline. Optuna studies are resumed by
# name (load_if_exists=True) keyed only on model/target/split/seed, with
# NO encoding of which data version produced them. Reusing an old
# RESULTS_DIR would silently resume trials whose objective values were
# computed on the old feature set / old target scale.
RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/dense_mlp")

# EDIT: "feature_set" values renamed to match data_utils.py's `dataset`
# argument, which matches the actual file stems written by
# Stage_4_Assembly/05_splits.ipynb (agg_means_{part}.parquet,
# agg_full_moments_{part}.parquet). "full_moments"/"means_only" no longer
# exist as names anywhere in the pipeline.
DATASETS     = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]

# ── Reproducibility seeds ──
SEEDS = [42, 123, 456]

N_TRIALS = 40

# ── Load Huber deltas once, keyed off split name and target family ──
# 'market' is the aggregate-side key in huber_delta.json (04_targets.ipynb
# computes deltas for 'market' and 'panel' separately). This notebook only
# ever touches agg_means/agg_full_moments, so always 'market'.
with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]


# ═══════════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY
# ═══════════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    """Set all random seeds for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ═══════════════════════════════════════════════════════════════════════════════
# ARCHITECTURE — widths derived per dataset from the taxonomy
# ═══════════════════════════════════════════════════════════════════════════════
# EDIT: previously N_SUBTHEMES/N_THEMES were fixed module constants shared
# across both feature sets. They are now looked up per dataset via the
# taxonomy CSV, using EXACTLY the same source of truth
# (data_utils.load_theme_assignment) that SparseKAN.from_taxonomy() and
# SparseMLP.from_taxonomy() use to build their masks -- so the dense model's
# hidden widths always match the sparse model's subtheme/theme counts for
# the same dataset, keeping the "matched topology" comparison honest even
# if the taxonomy changes again later.

def get_architecture_widths(dataset):
    """
    Returns (n_subthemes, n_themes) for a given dataset, read from its
    taxonomy file. Restricted to columns actually present in the taxonomy
    file -- for agg_means this may be a strict subset of the dataset's own
    features if a base factor genuinely has no taxonomy row, though
    08_preflight.ipynb Check 1 should already guarantee zero such rows.
    """
    tax = load_theme_assignment(dataset, THEMES_DIR)
    n_subthemes = tax["subtheme_id"].nunique()
    n_themes    = tax["theme_id"].nunique()
    return n_subthemes, n_themes


ARCHITECTURE = {ds: get_architecture_widths(ds) for ds in DATASETS}

print("=" * 70)
print("  ARCHITECTURE WIDTHS (derived from taxonomy, per dataset)")
print("=" * 70)
for ds, (n_sub, n_th) in ARCHITECTURE.items():
    print(f"  {ds:<20} n_subthemes={n_sub:>4}   n_themes={n_th:>3}")


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════════════════════════

def make_dense_mlp(n_features, n_subthemes, n_themes):
    """
    Dense MLP: [n_features, n_subthemes, n_themes, 1] with SiLU activation.

    This is exactly what the Dense KAN computes when all spline
    coefficients are zero — the base_weight * SiLU(x) pathway.
    Any performance difference between MLP and KAN is purely
    attributable to the B-spline components.
    """
    return nn.Sequential(
        nn.Linear(n_features, n_subthemes),
        nn.SiLU(),
        nn.Linear(n_subthemes, n_themes),
        nn.SiLU(),
        nn.Linear(n_themes, 1),
    )


# ═══════════════════════════════════════════════════════════════════════════════
# L1 REGULARISATION
# ═══════════════════════════════════════════════════════════════════════════════

def mlp_weight_l1(model):
    """L1 on all Linear layer weight matrices."""
    loss = torch.tensor(0.0, device=next(model.parameters()).device)
    for module in model.modules():
        if isinstance(module, nn.Linear):
            loss = loss + module.weight.abs().sum()
    return loss


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL FACTORY FOR OPTUNA
# ═══════════════════════════════════════════════════════════════════════════════

def make_model_factory(n_features, n_subthemes, n_themes, huber_delta, target_type):
    """
    Factory for Optuna. Same 4 hyperparameters as the Dense KAN:
        lr:           [1e-4, 1e-2]    log
        weight_decay: [1e-6, 1e-2]    log
        batch_size:   [64, 128, 256]  categorical
        reg_weight:   [1e-6, 1e-2]    log

    huber_delta is a FIXED value (not tuned), the per-split value from
    huber_delta.json, passed through to every trial's train_kwargs. Only
    used when target_type == "continuous" (get_criterion ignores it for
    binary).
    """
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])
        reg_weight   = trial.suggest_float("reg_weight", 1e-6, 1e-2, log=True)

        model = make_dense_mlp(n_features, n_subthemes, n_themes)

        train_kwargs = {
            "lr":           lr,
            "weight_decay": weight_decay,
            "reg_fn":       mlp_weight_l1,
            "reg_weight":   reg_weight,
            "n_epochs":     300,
            "patience":     20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta

        return model, train_kwargs

    return factory


# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE EXPERIMENT
# ═══════════════════════════════════════════════════════════════════════════════

def run_single_experiment(split_name, dataset, target_type, device,
                          seed, seed_results_dir):
    """
    Run Optuna search, retrain with best params, evaluate, save everything.

    Each seed gets:
      - Its own Optuna study (seeded TPE sampler → reproducible search)
      - Its own checkpoint, predictions, and metrics subfolder
      - Independent best_params (may differ across seeds)

    Optuna direction is always "maximize":
        binary     → maximise val AUC
        continuous → maximise val R²
    """
    model_name = f"dense_mlp_{dataset}"

    print(f"\n{'─'*60}")
    print(f"  seed={seed} / {dataset} / {split_name} / {target_type}")
    print(f"{'─'*60}")

    data       = load_split(split_name, dataset, SPLITS_DIR)
    n_features = data["n_features"]
    n_subthemes, n_themes = ARCHITECTURE[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None

    # ── OPTUNA SEARCH (seeded TPE sampler) ──
    factory     = make_model_factory(n_features, n_subthemes, n_themes,
                                     huber_delta, target_type)
    metric_name = "AUC" if target_type == "binary" else "R²"

    study_path = (seed_results_dir / "optuna"
                  / f"{model_name}_{target_type}_{split_name}.db")
    study_path.parent.mkdir(parents=True, exist_ok=True)

    study = optuna.create_study(
        study_name=f"{model_name}_{target_type}_{split_name}_seed{seed}",
        storage=f"sqlite:///{study_path}",
        direction="maximize",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=seed),
    )

    def objective(trial):
        model, train_kwargs = factory(trial)
        batch_size = trial.params["batch_size"]

        loaders = get_dataloaders(
            split_name, dataset, SPLITS_DIR,
            target_type=target_type,
            batch_size=batch_size,
        )

        result = train_model(
            model=model,
            train_loader=loaders["train"],
            val_loader=loaders["val"],
            device=device,
            target_type=target_type,
            pos_weight=pos_weight if target_type == "binary" else None,
            verbose=False,
            **train_kwargs,
        )

        return result["best_val_metric"]  # AUC (binary) or R² (continuous)

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    existing  = sum(1 for t in study.trials
                    if t.state == optuna.trial.TrialState.COMPLETE)
    remaining = max(0, N_TRIALS - existing)

    if remaining > 0:
        print(f"  Running {remaining} Optuna trials ({existing} already complete)")
        study.optimize(objective, n_trials=remaining, show_progress_bar=True)
    else:
        print(f"  Study already has {existing} completed trials — skipping Optuna")

    best_params = study.best_params
    print(f"  Optuna best {metric_name}: {study.best_value:.4f}")
    print(f"  Best params: {best_params}")

    # ── FINAL TRAINING with best params ──
    batch_size = best_params.get("batch_size", 128)
    loaders    = get_dataloaders(
        split_name, dataset, SPLITS_DIR,
        target_type=target_type,
        batch_size=batch_size,
    )

    model = make_dense_mlp(n_features, n_subthemes, n_themes)

    final_train_kwargs = dict(
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        reg_fn=mlp_weight_l1,
        reg_weight=best_params["reg_weight"],
        pos_weight=pos_weight if target_type == "binary" else None,
        n_epochs=300,
        patience=20,
        verbose=True,
        log_every=20,
    )
    if target_type == "continuous":
        final_train_kwargs["huber_delta"] = huber_delta

    result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        target_type=target_type,
        **final_train_kwargs,
    )

    # ── EVALUATE on train/val/test ──
    # For continuous: pass y_true_binary explicitly so derive_binary_from_continuous
    # receives correct 0/1 labels (DataLoader carries minret_5d_pct, not binary labels).
    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        if target_type == "binary":
            cal = compute_calibration(metrics["y_true"], metrics["y_prob"])
            metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            dates=data[f"dates_{part}"],
            returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters={**best_params, "seed": seed,
                             "huber_delta": huber_delta} if part == "test" else None,
            results_dir=seed_results_dir,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        all_metrics[part] = metrics

    # ── SAVE CHECKPOINT ──
    ckpt_dir = seed_results_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model,
        train_result=result,
        hyperparameters={**best_params, "seed": seed, "huber_delta": huber_delta},
        model_config={
            "type":        "DenseMLP",
            "dataset":     dataset,
            "target_type": target_type,
            "layers":      [n_features, n_subthemes, n_themes, 1],
            "activation":  "SiLU",
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}.pt",
    )

    # ── PRINT SUMMARY ──
    if target_type == "binary":
        print(f"\n  Results (seed={seed}):")
        print(f"    Train AUC: {all_metrics['train']['auc']:.4f}")
        print(f"    Val AUC:   {all_metrics['val']['auc']:.4f}")
        print(f"    Test AUC:  {all_metrics['test']['auc']:.4f}")
        print(f"    Gap:       "
              f"{all_metrics['train']['auc'] - all_metrics['test']['auc']:+.4f}")
    else:
        print(f"\n  Results (seed={seed}, huber_delta={huber_delta:.4f}):")
        print(f"    Train R²:    {all_metrics['train']['r2']:.4f}  "
              f"(MSE={all_metrics['train']['mse']:.4f})")
        print(f"    Val R²:      {all_metrics['val']['r2']:.4f}  "
              f"(MSE={all_metrics['val']['mse']:.4f})")
        print(f"    Test R²:     {all_metrics['test']['r2']:.4f}  "
              f"(MSE={all_metrics['test']['mse']:.4f})")
        print(f"    Derived AUC: {all_metrics['test']['derived_auc']:.4f}")
        print(f"    Pred std:    {all_metrics['test']['pred_std']:.4f}  "
              f"(sanity: minret_5d_pct is a percentage, so std should "
              f"clearly exceed 0.01 -- treat near-zero as constant output)")

    return {
        "best_params": best_params,
        "metrics":     all_metrics,
        "best_epoch":  result["best_epoch"],
        "total_time":  result["total_time"],
    }


# %% [markdown]
# ## Run All Experiments (3 Seeds × 16 Configurations)

# %%
device     = get_device()
configs    = len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)  # 16
total_runs = len(SEEDS) * configs                                   # 48

n_params_by_dataset = {}
for ds in DATASETS:
    n_feat = load_split(ALL_SPLITS[0], ds, SPLITS_DIR)["n_features"]
    n_sub, n_th = ARCHITECTURE[ds]
    n_params_by_dataset[ds] = sum(
        p.numel() for p in make_dense_mlp(n_feat, n_sub, n_th).parameters()
    )

print("=" * 70)
print(f"  DENSE MLP: 3 Seeds × 4 Splits × 2 Datasets × 2 Targets = {total_runs} runs")
print(f"  Seeds: {SEEDS}")
print(f"  Activation: SiLU")
for ds in DATASETS:
    n_sub, n_th = ARCHITECTURE[ds]
    print(f"  {ds:<20} widths=[n_feat, {n_sub}, {n_th}, 1]   "
          f"params={n_params_by_dataset[ds]:,}")
print(f"  Loss: binary=BCEWithLogitsLoss, "
      f"continuous=HuberLoss(delta=per-split, see huber_delta.json)")
print(f"  Early stop: binary=AUC, continuous=R²  (both maximize)")
print(f"  Optuna: {N_TRIALS} trials per run, seeded TPE sampler")
print(f"  Regularisation: L1 on weight matrices")
print(f"  Results saving to: {RESULTS_DIR}/seed_*/")
print("=" * 70)

all_results       = []
best_params_store = {}
completed         = 0
failed            = 0
total_start       = time.time()

for seed in SEEDS:

    # ── Set all random seeds for this pipeline run ──
    set_seed(seed)
    seed_results_dir = RESULTS_DIR / f"seed_{seed}"

    print(f"\n\n{'═'*70}")
    print(f"  SEED {seed} — saving to {seed_results_dir}")
    print(f"{'═'*70}")

    for dataset in DATASETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                try:
                    exp = run_single_experiment(
                        split_name, dataset, target_type, device,
                        seed=seed,
                        seed_results_dir=seed_results_dir,
                    )

                    all_results.append({
                        "seed":    seed,
                        "dataset": dataset,
                        "split":   split_name,
                        "target":  target_type,
                        **{f"test_{k}": v for k, v in exp["metrics"]["test"].items()
                           if not isinstance(v, np.ndarray)},
                        "best_epoch": exp["best_epoch"],
                        "time_s":     exp["total_time"],
                    })

                    key = (seed, dataset, target_type, split_name)
                    best_params_store[key] = exp["best_params"]
                    completed += 1

                    elapsed = time.time() - total_start
                    rate = elapsed / completed
                    remaining_est = rate * (total_runs - completed)
                    print(f"\n  ✓ Completed {completed}/{total_runs}  "
                          f"({elapsed/60:.0f}min elapsed, "
                          f"~{remaining_est/60:.0f}min remaining)")

                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): seed={seed} "
                          f"{dataset}/{split_name}/{target_type}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
print(f"  Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"{'='*70}")


# %% [markdown]
# ## Cross-Seed Summary
#
# The key output: mean ± std across the 3 seeds for each configuration.
# This tells us how stable the full pipeline is to random initialisation.

# %%
if all_results:
    results_df = pd.DataFrame(all_results)

    # ── Save raw results CSV (all seeds, all configs) ──
    raw_path = RESULTS_DIR / "mlp_all_seeds_raw.csv"
    results_df.to_csv(raw_path, index=False)
    print(f"  Raw results saved to {raw_path}")

    # ═══════════════════════════════════════════════════════════════════════
    # BINARY AUC — per split (mean ± std across seeds)
    # ═══════════════════════════════════════════════════════════════════════
    binary_df = results_df[results_df["target"] == "binary"]

    print("\n" + "=" * 70)
    print("  DENSE MLP — Binary Test AUC (mean ± std across 3 seeds)")
    print("=" * 70 + "\n")

    if "test_auc" in binary_df.columns and len(binary_df) > 0:
        agg = binary_df.groupby(["dataset", "split"])["test_auc"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")

        grand = binary_df.groupby("dataset")["test_auc"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    # ═══════════════════════════════════════════════════════════════════════
    # CONTINUOUS R² — per split (mean ± std across seeds)
    # ═══════════════════════════════════════════════════════════════════════
    cont_df = results_df[results_df["target"] == "continuous"]

    print("\n" + "=" * 70)
    print("  DENSE MLP — Continuous Test R² (mean ± std across 3 seeds)")
    print("=" * 70 + "\n")

    if "test_r2" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_r2"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")

        grand = cont_df.groupby("dataset")["test_r2"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    # ═══════════════════════════════════════════════════════════════════════
    # CONTINUOUS DERIVED AUC — per split (mean ± std across seeds)
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  DENSE MLP — Continuous Derived AUC (mean ± std across 3 seeds)")
    print("=" * 70 + "\n")

    if "test_derived_auc" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_derived_auc"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")

        grand = cont_df.groupby("dataset")["test_derived_auc"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    # ═══════════════════════════════════════════════════════════════════════
    # CONTINUOUS MSE — per split (mean ± std across seeds)
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  DENSE MLP — Continuous Test MSE (mean ± std across 3 seeds)")
    print("=" * 70 + "\n")

    if "test_mse" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_mse"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")

        grand = cont_df.groupby("dataset")["test_mse"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())
        print(f"\n  NOTE: MSE is in minret_5d_pct units (percentage points "
              f"squared), NOT z-score units -- do not compare these MSE "
              f"values against any figure computed under the old pipeline.")

    # ═══════════════════════════════════════════════════════════════════════
    # PREDICTION STD SANITY CHECK
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  CONTINUOUS SANITY: Prediction std")
    print("  minret_5d_pct is a raw percentage; a healthy model's predictions")
    print("  should show meaningfully more than a trivial constant. Flag any")
    print("  run whose pred_std looks suspiciously close to zero relative to")
    print("  the others -- there is no longer a single fixed 0.01 threshold")
    print("  since this is no longer a pre-standardised z-score.")
    print("=" * 70 + "\n")

    if "test_pred_std" in cont_df.columns:
        for _, row in cont_df.iterrows():
            print(f"  seed={row['seed']}  {row['dataset']:<20} "
                  f"{row['split']:<10}  "
                  f"pred_std={row.get('test_pred_std', 0):.4f}")

    # ═══════════════════════════════════════════════════════════════════════
    # SEED STABILITY DIAGNOSTIC
    # How much do the best hyperparameters vary across seeds?
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  SEED STABILITY: Best hyperparameters across seeds")
    print("=" * 70 + "\n")

    print(f"  {'Seed':>5} {'Dataset':<20} {'Target':<12} {'Split':<10} "
          f"{'lr':>10} {'wd':>10} {'bs':>5} {'reg_w':>10}")
    print("  " + "-" * 85)
    for (seed, ds, tt, split), params in sorted(best_params_store.items()):
        print(f"  {seed:>5} {ds:<20} {tt:<12} {split:<10} "
              f"{params['lr']:>10.6f} {params['weight_decay']:>10.6f} "
              f"{params['batch_size']:>5} {params['reg_weight']:>10.2e}")

    # ═══════════════════════════════════════════════════════════════════════
    # SEED VARIANCE SUMMARY
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  SEED VARIANCE SUMMARY")
    print("=" * 70 + "\n")

    for tt in TARGET_TYPES:
        metric_col   = "test_auc" if tt == "binary" else "test_derived_auc"
        metric_label = "AUC"      if tt == "binary" else "Derived AUC"
        subset = results_df[results_df["target"] == tt]

        if metric_col not in subset.columns or len(subset) == 0:
            continue

        print(f"  {tt.upper()} ({metric_label}):")
        per_config = subset.groupby(["dataset", "split"])[metric_col].agg(
            ["mean", "std"]
        )
        max_std  = per_config["std"].max()
        mean_std = per_config["std"].mean()
        print(f"    Mean seed std across configs: {mean_std:.4f}")
        print(f"    Max seed std across configs:  {max_std:.4f}")

        if max_std < 0.01:
            print(f"    → Very stable: seed choice barely matters")
        elif max_std < 0.03:
            print(f"    → Moderately stable: some sensitivity to initialisation")
        else:
            print(f"    → High variance: results depend substantially on seed")
        print()

    # ═══════════════════════════════════════════════════════════════════════
    # TIMING
    # ═══════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("  TIMING")
    print("=" * 70 + "\n")

    for _, row in results_df.iterrows():
        print(f"  seed={row['seed']}  {row['dataset']:<20} "
              f"{row['split']:<10} {row['target']:<12} "
              f"best_epoch={row['best_epoch']:>3}  {row['time_s']:>6.1f}s")

    # ═══════════════════════════════════════════════════════════════════════
    # SAVE AGGREGATED SUMMARY CSV
    # ═══════════════════════════════════════════════════════════════════════
    summary_rows = []
    for tt in TARGET_TYPES:
        subset = results_df[results_df["target"] == tt]
        if len(subset) == 0:
            continue

        metric_cols = [c for c in subset.columns
                       if c.startswith("test_") and
                       subset[c].dtype in [np.float64, np.float32, float]]

        agg = subset.groupby(["dataset", "split"])[metric_cols].agg(
            ["mean", "std"]
        ).reset_index()

        agg.columns = [
            f"{c[0]}_{c[1]}" if c[1] else c[0]
            for c in agg.columns
        ]

        agg["target"] = tt
        summary_rows.append(agg)

    if summary_rows:
        summary_df  = pd.concat(summary_rows, ignore_index=True)
        summary_path = RESULTS_DIR / "mlp_cross_seed_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"\n  Cross-seed summary saved to {summary_path}")

else:
    print("\n  No results to display — all experiments failed.")


# %% [markdown]
# ## File Inventory

# %%
print("\n" + "=" * 70)
print("  SAVED FILES (on Google Drive)")
print("=" * 70)

for seed in SEEDS:
    seed_dir = RESULTS_DIR / f"seed_{seed}"
    print(f"\n  ── seed_{seed}/ ──")
    for subdir in ["predictions", "metrics", "checkpoints", "optuna"]:
        d = seed_dir / subdir
        if d.exists():
            files = list(d.glob("dense_mlp_*"))
            print(f"    {subdir}/: {len(files)} files")
        else:
            print(f"    {subdir}/: (not yet created)")

for fname in ["mlp_all_seeds_raw.csv", "mlp_cross_seed_summary.csv"]:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        print(f"\n  {fname}: ✓")
    else:
        print(f"\n  {fname}: (not yet created)")


# %% [markdown]
# ## Backtests (Seed-Averaged Signal)
#
# Predictions are averaged across the 3 seeds before backtesting.
# Averaging reduces noise and produces a single ensemble signal —
# this is both cleaner to interpret and what you would do in practice.
#
# One `run_full_backtest` call per dataset × split × target_type:
#   - Signal diagnostics (raw vs smoothed range)
#   - Simple timing: threshold chosen on val (net-of-cost Sortino)
#   - Simple timing cost sensitivity sweep
#   - Asymmetric risk-scaled: params found on val
#   - Risk-scaled cost sensitivity sweep
#   - Side-by-side comparison vs buy-and-hold
#
# Binary signal  : averaged y_prob  (go_cash_when="above")
# Continuous signal: averaged y_pred (go_cash_when="below")
#
# EDIT: results are now SAVED to disk, not just printed. Two files:
#   backtests/backtest_summary.csv       flat, one row per (dataset, split,
#                                         target_type, strategy) -- quick to
#                                         pivot/plot from later
#   backtests/backtest_full_results.json full nested dict per config,
#                                         including cost-sensitivity tables
#                                         and the chosen risk-scaled params --
#                                         needed if you ever want to see WHY
#                                         a threshold was picked, not just
#                                         the final Sharpe
#
# NOTE: evaluation.py's fixed continuous threshold grids
# (find_best_threshold / find_best_risk_scaled_params) still assume a
# roughly [-5, +2] z-score range from the old target. minret_5d_pct lives
# on a different scale, so these grids may sit partly or entirely outside
# where the model's actual predictions land, biasing the backtest toward
# a grid edge. This is a KNOWN, DEFERRED issue in evaluation.py -- not
# something to fix in this notebook. If backtest thresholds print a
# grid-edge warning here, that is this known issue surfacing, not a new bug.

# %%
def load_averaged_predictions(model_name, split_name, target_type, part,
                               seeds, results_dir):
    """
    Load predictions from each seed folder and average the signal.

    Binary:     averages y_prob across seeds
    Continuous: averages y_pred across seeds

    Returns (returns, avg_signal) as numpy arrays.
    Daily returns are identical across seeds (same data), so we take
    them from the first seed only.
    """
    signals = []
    returns = None

    for seed in seeds:
        seed_dir = results_dir / f"seed_{seed}"
        loaded = load_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            results_dir=seed_dir,
        )
        preds = loaded["predictions"]

        if returns is None:
            returns = preds["daily_return"].values

        if target_type == "binary":
            signals.append(preds["y_prob"].values)
        else:
            signals.append(preds["y_pred"].values)

    avg_signal = np.mean(np.stack(signals, axis=0), axis=0)
    return returns, avg_signal


def _make_json_safe(obj):
    """Recursively convert DataFrames/numpy types to plain JSON-serialisable types."""
    if isinstance(obj, dict):
        return {k: _make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame):
        return obj.reset_index().to_dict(orient="records")
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# %%
print("\n" + "=" * 70)
print("  DENSE MLP — BACKTESTS (seed-averaged signal)")
print("  Signal = mean prediction across seeds 42, 123, 456")
print("=" * 70)

backtest_dir = RESULTS_DIR / "backtests"
backtest_dir.mkdir(parents=True, exist_ok=True)

backtest_rows    = []   # flat summary row per (dataset, split, target_type, strategy)
backtest_records = {}   # full nested dict, saved as one JSON for deep inspection

for dataset in DATASETS:
    for split_name in ALL_SPLITS:
        model_name = f"dense_mlp_{dataset}"

        # ── Binary ──
        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "binary", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "binary", "test", SEEDS, RESULTS_DIR)

        bt_binary = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="above",
            model_name=f"{model_name} (binary)",
            split_name=split_name,
        )

        # ── Continuous ──
        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "continuous", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "continuous", "test", SEEDS, RESULTS_DIR)

        bt_continuous = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="below",
            model_name=f"{model_name} (continuous)",
            split_name=split_name,
        )

        # ── Store the full nested result for both target types ──
        key = f"{dataset}/{split_name}"
        backtest_records[key] = {
            "binary": bt_binary,
            "continuous": bt_continuous,
        }

        # ── Flat summary rows for a quick-scan CSV ──
        for target_type, bt in [("binary", bt_binary), ("continuous", bt_continuous)]:
            for strategy_name, strategy_key in [("simple", "simple"), ("risk_scaled", "risk_scaled")]:
                bt_result = bt[strategy_key]
                backtest_rows.append({
                    "dataset": dataset,
                    "split": split_name,
                    "target_type": target_type,
                    "strategy": strategy_name,
                    "sharpe": bt_result["sharpe"],
                    "sortino": bt_result["sortino"],
                    "annual_return": bt_result["annual_return"],
                    "max_drawdown": bt_result["max_drawdown"],
                    "cumulative_return": bt_result["cumulative_return"],
                    "avg_exposure": bt_result["avg_exposure"],
                    "annual_turnover": bt_result["annual_turnover"],
                    "buy_hold_sharpe": bt_result["buy_hold_sharpe"],
                    "buy_hold_sortino": bt_result["buy_hold_sortino"],
                    "buy_hold_cumulative": bt_result["buy_hold_cumulative"],
                })

# ── Save flat summary CSV (quick to scan/pivot) ──
backtest_summary_df = pd.DataFrame(backtest_rows)
backtest_summary_path = backtest_dir / "backtest_summary.csv"
backtest_summary_df.to_csv(backtest_summary_path, index=False)
print(f"\n  Backtest summary saved to {backtest_summary_path}")

# ── Save full nested results as JSON (includes cost-sensitivity tables, chosen params, etc.) ──
backtest_json_path = backtest_dir / "backtest_full_results.json"
with open(backtest_json_path, "w") as f:
    json.dump(_make_json_safe(backtest_records), f, indent=2, default=str)
print(f"  Full backtest results (incl. cost-sensitivity tables) saved to {backtest_json_path}")


# %% [markdown]
# ## Disconnect Runtime

# %%
print("All experiments complete. Disconnecting runtime...")
from google.colab import runtime
runtime.unassign()

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
  ARCHITECTURE WIDTHS (derived from taxonomy, per dataset)
  agg_full_moments     n_subthemes= 331   n_themes= 13
  agg_means            n_subthemes= 128   n_themes= 13
Device: Tesla T4 (CUDA)
  DENSE MLP: 3 Seeds × 4 Splits × 2 Datasets × 2 Targets = 48 runs
  Seeds: [42, 123, 456]
  Activation: SiLU
  agg_full_moments     widths=[n_feat, 331, 13, 1]   params=567,030
  agg_means            widths=[n_feat, 128, 13, 1]   params=75,291
  Loss: binary=BCEWithLogitsLoss, continuous=HuberLoss(delta=per-split, see huber_delta.json)
  Early stop: binary=AUC, continuous=R²  (both maximize)
  Optuna: 40 trials per run, seeded TPE sampler
  Regularisation: L1 on weight matrices
  Results saving to: /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/dense_mlp/seed_*/


══════════════════════════════════════════════════════════════════════
  SEED 42 — saving to /content/drive/MyDrive/Thesis/Data/Results/Dense_v

[I 2026-08-14 01:11:25,264] A new study created in RDB with name: dense_mlp_agg_full_moments_binary_Split_A_seed42


  Running 40 Optuna trials (0 already complete)


  0%|          | 0/40 [00:00<?, ?it/s]

  Optuna best AUC: 0.8163
  Best params: {'lr': 0.00017393415293642963, 'weight_decay': 4.374029324858844e-05, 'batch_size': 128, 'reg_weight': 0.009272085083831887}
  Epoch    1 | Train loss 1.0997 | Val loss 0.7401  AUC 0.6722 | LR 1.7e-04
  Epoch   20 | Train loss 0.9619 | Val loss 0.7025  AUC 0.7900 | LR 1.7e-04
  Epoch   40 | Train loss 0.9319 | Val loss 0.6577  AUC 0.7977 | LR 1.7e-04
  Early stop at epoch 55. Best val AUC: 0.8009 at epoch 35
  Training complete in 4.6s

  Results (seed=42):
    Train AUC: 0.8026
    Val AUC:   0.8009
    Test AUC:  0.7781
    Gap:       +0.0245

  ✓ Completed 1/48  (3min elapsed, ~143min remaining)

────────────────────────────────────────────────────────────
  seed=42 / agg_full_moments / Split_B / binary
────────────────────────────────────────────────────────────
  Running 40 Optuna trials (0 already complete)


...

  Optuna best R²: 0.1213
  Best params: {'lr': 0.004630773388274391, 'weight_decay': 2.0902372326962702e-05, 'batch_size': 256, 'reg_weight': 0.00994281556009699}
  Epoch    1 | Train Huber 0.5996 | Val Huber 0.6011  MSE 1.2487  R² -0.3033 | LR 4.6e-03
  Epoch   20 | Train Huber 0.3195 | Val Huber 0.5170  MSE 1.0684  R² -0.1152 | LR 2.3e-03
  Early stop at epoch 32. Best val R²: 0.0261 at epoch 12
  Training complete in 1.6s

  Results (seed=456, huber_delta=2.4735):
    Train R²:    0.5614  (MSE=0.6674)
    Val R²:      0.0261  (MSE=0.9331)
    Test R²:     0.2667  (MSE=2.0548)
    Derived AUC: 0.7313
    Pred std:    0.6894  (sanity: minret_5d_pct is a percentage, so std should clearly exceed 0.01 -- treat near-zero as constant output)

  ✓ Completed 46/48  (129min elapsed, ~6min remaining)

────────────────────────────────────────────────────────────
  seed=456 / agg_means / Split_C / continuous
────────────────────────────────────────────────────────────
  Running 40 Optuna trials 

  0%|          | 0/40 [00:00<?, ?it/s]

  Optuna best R²: 0.3543
  Best params: {'lr': 0.0068042121651215705, 'weight_decay': 0.009577492041893492, 'batch_size': 64, 'reg_weight': 0.002758382363478075}
  Epoch    1 | Train Huber 0.4459 | Val Huber 0.7665  MSE 1.9890  R² 0.2956 | LR 6.8e-03
  Epoch   20 | Train Huber 0.2749 | Val Huber 0.8562  MSE 2.3905  R² 0.1534 | LR 1.7e-03
  Early stop at epoch 23. Best val R²: 0.3385 at epoch 3
  Training complete in 3.6s

  Results (seed=456, huber_delta=2.4230):
    Train R²:    0.4806  (MSE=0.7443)
    Val R²:      0.3385  (MSE=1.8679)
    Test R²:     0.1278  (MSE=0.8843)
    Derived AUC: 0.7974
    Pred std:    0.5439  (sanity: minret_5d_pct is a percentage, so std should clearly exceed 0.01 -- treat near-zero as constant output)

  ✓ Completed 47/48  (132min elapsed, ~3min remaining)

────────────────────────────────────────────────────────────
  seed=456 / agg_means / Split_D / continuous
────────────────────────────────────────────────────────────
  Running 40 Optuna trials (0 a

  0%|          | 0/40 [00:00<?, ?it/s]

  Optuna best R²: 0.2125
  Best params: {'lr': 0.005976244953040927, 'weight_decay': 5.2814344185348584e-05, 'batch_size': 64, 'reg_weight': 0.0033377187536662235}
  Epoch    1 | Train Huber 0.5085 | Val Huber 0.4788  MSE 0.9614  R² 0.1445 | LR 6.0e-03
  Epoch   20 | Train Huber 0.3305 | Val Huber 0.6231  MSE 1.2542  R² -0.1161 | LR 1.5e-03
  Early stop at epoch 21. Best val R²: 0.1445 at epoch 1
  Training complete in 3.8s

  Results (seed=456, huber_delta=2.4998):
    Train R²:    0.4279  (MSE=0.9769)
    Val R²:      0.1445  (MSE=0.9614)
    Test R²:     -0.1437  (MSE=0.5668)
    Derived AUC: 0.5858
    Pred std:    0.3060  (sanity: minret_5d_pct is a percentage, so std should clearly exceed 0.01 -- treat near-zero as constant output)

  ✓ Completed 48/48  (134min elapsed, ~0min remaining)


  FINISHED: 48/48 completed, 0 failed
  Total time: 133.9 minutes (2.2 hours)
  Raw results saved to /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/dense_mlp/mlp_all_seeds_raw

In [1]:
# %% [markdown]
# # 05b — Dense MLP Pruning Sweep
#
# Same protocol as 02b (Dense KAN pruning sweep) but for the MLP.
# Tests whether aggressive L1 weight pruning can make the dense MLP
# train properly, and compares the pruning curve to the Dense KAN's.
#
# At matched sparsity, if the KAN curve sits above the MLP curve,
# B-splines add value. If they overlap, splines add nothing beyond SiLU.
#
# 8 reg_weights × 4 splits × 2 feature sets × 2 targets = 128 runs
# Each run: 20 Optuna trials. Estimated ~4-5 hours on local CPU.

# %%
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import time
import torch
import torch.nn as nn
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device
from training import train_model, save_checkpoint
from evaluation import evaluate_model, save_predictions, compute_calibration

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

SPLITS_DIR = Path("../../..") / "Data" / "Splits"
RESULTS_DIR = Path("../../..") / "Data" / "Results" / "Dense_vs_Sparse_KAN" / "pruning_sweep_mlp"

FEATURE_SETS = ["full_moments", "means_only"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS = ["Split_A", "Split_B", "Split_C", "Split_D"]

# ── Fixed reg_weight values (same as KAN sweep for direct comparison) ──
REG_WEIGHTS = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

# ── Fixed architecture (matches KAN layer widths) ──
N_SUBTHEMES = 99
N_THEMES = 13

# ── Optuna settings (3 params, 20 trials) ──
N_TRIALS = 20

# ── Connection survival threshold ──
WEIGHT_THRESHOLD = 0.01


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════════════════════════

def make_dense_mlp(n_features):
    """Dense MLP: [n_features, 99, 13, 1] with SiLU. ~220K params."""
    return nn.Sequential(
        nn.Linear(n_features, N_SUBTHEMES),
        nn.SiLU(),
        nn.Linear(N_SUBTHEMES, N_THEMES),
        nn.SiLU(),
        nn.Linear(N_THEMES, 1),
    )


# ═══════════════════════════════════════════════════════════════════════════════
# L1 ON WEIGHT MATRICES
# ═══════════════════════════════════════════════════════════════════════════════

def mlp_weight_l1(model):
    """L1 on all Linear layer weight matrices."""
    loss = torch.tensor(0.0, device=next(model.parameters()).device)
    for module in model.modules():
        if isinstance(module, nn.Linear):
            loss = loss + module.weight.abs().sum()
    return loss


# ═══════════════════════════════════════════════════════════════════════════════
# CONNECTION COUNTING
# ═══════════════════════════════════════════════════════════════════════════════

def count_surviving_connections(model, threshold=WEIGHT_THRESHOLD):
    """
    Count weights where |weight[i,j]| > threshold.
    Analogous to count_surviving_edges for the KAN.
    """
    layer_stats = []
    total_alive = 0
    total_connections = 0
    layer_idx = 0

    for module in model.modules():
        if isinstance(module, nn.Linear):
            alive_mask = module.weight.abs() > threshold
            n_alive = alive_mask.sum().item()
            n_total = alive_mask.numel()

            layer_stats.append({
                "layer": layer_idx,
                "alive": int(n_alive),
                "total": int(n_total),
                "pct_alive": n_alive / n_total * 100 if n_total > 0 else 0,
                "shape": f"{module.in_features}→{module.out_features}",
            })

            total_alive += n_alive
            total_connections += n_total
            layer_idx += 1

    return {
        "total_alive": total_alive,
        "total_connections": total_connections,
        "pct_alive": total_alive / total_connections * 100 if total_connections > 0 else 0,
        "layers": layer_stats,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL FACTORY (reg_weight NOT in Optuna)
# ═══════════════════════════════════════════════════════════════════════════════

def make_model_factory(n_features, fixed_reg_weight):
    """
    Factory for Optuna. Searches only 3 hyperparameters:
        lr:           [1e-4, 1e-2]    log
        weight_decay: [1e-6, 1e-2]    log
        batch_size:   [64, 128, 256]  categorical

    reg_weight is FIXED — it's the variable under investigation.
    """
    def factory(trial):
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])

        model = make_dense_mlp(n_features)

        train_kwargs = {
            "lr": lr,
            "weight_decay": weight_decay,
            "reg_fn": mlp_weight_l1,
            "reg_weight": fixed_reg_weight,
            "n_epochs": 300,
            "patience": 20,
        }

        return model, train_kwargs

    return factory


# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE EXPERIMENT
# ═══════════════════════════════════════════════════════════════════════════════

def run_pruning_experiment(split_name, feature_set, target_type, reg_weight, device):
    """Run one pruning experiment: Optuna → train → evaluate → count connections."""
    model_name = f"dense_mlp_pruned_{feature_set}"
    rw_str = f"rw{reg_weight:.4f}".replace(".", "p")

    print(f"\n{'─'*60}")
    print(f"  {feature_set} / {split_name} / {target_type} / reg_weight={reg_weight}")
    print(f"{'─'*60}")

    # ── Load data ──
    data = load_split(split_name, feature_set, SPLITS_DIR)
    n_features = data["n_features"]

    n_pos = data["y_train"].sum()
    n_neg = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    # ── OPTUNA SEARCH ──
    factory = make_model_factory(n_features, reg_weight)
    direction = "maximize" if target_type == "binary" else "minimize"

    study_name = f"{model_name}_{target_type}_{split_name}_{rw_str}"
    study_path = RESULTS_DIR / "optuna" / f"{study_name}.db"
    study_path.parent.mkdir(parents=True, exist_ok=True)
    storage = f"sqlite:///{study_path}"

    study = optuna.create_study(
        study_name=study_name,
        storage=storage,
        direction=direction,
        load_if_exists=True,
    )

    def objective(trial):
        model, train_kwargs = factory(trial)
        batch_size = trial.params["batch_size"]

        loaders = get_dataloaders(
            split_name, feature_set, SPLITS_DIR,
            target_type=target_type,
            batch_size=batch_size,
        )

        result = train_model(
            model=model,
            train_loader=loaders["train"],
            val_loader=loaders["val"],
            device=device,
            target_type=target_type,
            pos_weight=pos_weight if target_type == "binary" else None,
            verbose=False,
            **train_kwargs,
        )

        return result["best_val_metric"]

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    existing = len([t for t in study.trials
                    if t.state == optuna.trial.TrialState.COMPLETE])
    remaining = max(0, N_TRIALS - existing)

    if remaining > 0:
        print(f"  Optuna: {remaining} trials ({existing} already complete)")
        study.optimize(objective, n_trials=remaining, show_progress_bar=True)
    else:
        print(f"  Optuna: {existing} trials already complete — skipping")

    best_params = study.best_params
    metric_name = "AUC" if target_type == "binary" else "MSE"
    print(f"  Best {metric_name}: {study.best_value:.4f}  params: {best_params}")

    # ── FINAL TRAINING ──
    batch_size = best_params.get("batch_size", 128)
    loaders = get_dataloaders(
        split_name, feature_set, SPLITS_DIR,
        target_type=target_type,
        batch_size=batch_size,
    )

    model = make_dense_mlp(n_features)

    result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        target_type=target_type,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        reg_fn=mlp_weight_l1,
        reg_weight=reg_weight,
        pos_weight=pos_weight if target_type == "binary" else None,
        n_epochs=300,
        patience=20,
        verbose=True,
        log_every=20,
    )

    # ── COUNT SURVIVING CONNECTIONS ──
    conn_stats = count_surviving_connections(model)
    print(f"\n  Surviving connections: {conn_stats['total_alive']:,} / "
          f"{conn_stats['total_connections']:,} ({conn_stats['pct_alive']:.1f}%)")
    for ls in conn_stats["layers"]:
        print(f"    Layer {ls['layer']} ({ls['shape']}): "
              f"{ls['alive']:,} / {ls['total']:,} ({ls['pct_alive']:.1f}%)")

    # ── EVALUATE ──
    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        if target_type == "binary":
            cal = compute_calibration(metrics["y_true"], metrics["y_prob"])
            metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=f"{model_name}_{rw_str}",
            split_name=split_name,
            target_type=target_type,
            part=part,
            dates=data[f"dates_{part}"],
            returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters={**best_params, "reg_weight": reg_weight} if part == "test" else None,
            results_dir=RESULTS_DIR,
        )

        all_metrics[part] = metrics

    # ── SAVE CHECKPOINT ──
    ckpt_dir = RESULTS_DIR / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model,
        train_result=result,
        hyperparameters={**best_params, "reg_weight": reg_weight},
        model_config={
            "layers": [n_features, N_SUBTHEMES, N_THEMES, 1],
            "activation": "SiLU",
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}_{rw_str}.pt",
    )

    # ── PRINT SUMMARY ──
    if target_type == "binary":
        print(f"  Test AUC: {all_metrics['test']['auc']:.4f}  "
              f"Train AUC: {all_metrics['train']['auc']:.4f}  "
              f"Gap: {all_metrics['train']['auc'] - all_metrics['test']['auc']:+.4f}")
    else:
        print(f"  Test R²: {all_metrics['test']['r2']:.4f}  "
              f"Derived AUC: {all_metrics['test'].get('derived_auc', 'N/A')}")

    return {
        "reg_weight": reg_weight,
        "feature_set": feature_set,
        "split": split_name,
        "target": target_type,
        "surviving_connections": conn_stats["total_alive"],
        "total_connections": conn_stats["total_connections"],
        "pct_alive": conn_stats["pct_alive"],
        "layer_stats": conn_stats["layers"],
        "best_params": best_params,
        "best_epoch": result["best_epoch"],
        "time_s": result["total_time"],
        "test_metrics": {k: v for k, v in all_metrics["test"].items()
                         if not isinstance(v, np.ndarray)},
        "train_metrics": {k: v for k, v in all_metrics["train"].items()
                          if not isinstance(v, np.ndarray)},
    }


# %% [markdown]
# ## Run Pruning Sweep

# %%
device = get_device()

total_runs = len(REG_WEIGHTS) * len(FEATURE_SETS) * len(TARGET_TYPES) * len(ALL_SPLITS)

n_params_full = sum(p.numel() for p in make_dense_mlp(2204).parameters())
n_params_means = sum(p.numel() for p in make_dense_mlp(714).parameters())

print("=" * 70)
print("  DENSE MLP PRUNING SWEEP")
print(f"  Architecture: [n_feat, {N_SUBTHEMES}, {N_THEMES}, 1] + SiLU")
print(f"  Parameters: full_moments={n_params_full:,}, means_only={n_params_means:,}")
print(f"  reg_weights: {REG_WEIGHTS}")
print(f"  Optuna: {N_TRIALS} trials per run (lr, weight_decay, batch_size)")
print(f"  Total runs: {total_runs}")
print(f"  Results saving to: {RESULTS_DIR}")
print("=" * 70)

all_results = []
completed = 0
failed = 0

total_start = time.time()

for feature_set in FEATURE_SETS:
    for target_type in TARGET_TYPES:
        for split_name in ALL_SPLITS:
            for reg_weight in REG_WEIGHTS:
                try:
                    exp = run_pruning_experiment(
                        split_name, feature_set, target_type, reg_weight, device,
                    )

                    all_results.append(exp)
                    completed += 1

                    elapsed = time.time() - total_start
                    rate = elapsed / completed
                    remaining_est = rate * (total_runs - completed)
                    print(f"\n  ✓ {completed}/{total_runs} complete  "
                          f"({elapsed/60:.0f}min elapsed, ~{remaining_est/60:.0f}min remaining)")

                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): {feature_set}/{split_name}/"
                          f"{target_type}/rw={reg_weight}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

            # ── Save pruning curve CSV after each complete sweep ──
            if all_results:
                curve_df = pd.DataFrame([
                    {
                        "reg_weight": r["reg_weight"],
                        "feature_set": r["feature_set"],
                        "split": r["split"],
                        "target": r["target"],
                        "surviving_connections": r["surviving_connections"],
                        "total_connections": r["total_connections"],
                        "pct_alive": r["pct_alive"],
                        "best_epoch": r["best_epoch"],
                        "time_s": r["time_s"],
                        **{f"test_{k}": v for k, v in r["test_metrics"].items()},
                        **{f"train_{k}": v for k, v in r["train_metrics"].items()},
                    }
                    for r in all_results
                ])
                curve_path = RESULTS_DIR / "pruning_curve_mlp.csv"
                curve_path.parent.mkdir(parents=True, exist_ok=True)
                curve_df.to_csv(curve_path, index=False)

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
print(f"  Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"{'='*70}")

# %% [markdown]
# ## Pruning Curve Summary

# %%
if all_results:
    curve_df = pd.read_csv(RESULTS_DIR / "pruning_curve_mlp.csv")

    # ── Binary AUC vs surviving connections ──
    print("\n" + "=" * 70)
    print("  PRUNING CURVE — Binary Test AUC vs Surviving Connections")
    print("=" * 70 + "\n")

    binary_curve = curve_df[curve_df["target"] == "binary"]
    if len(binary_curve) > 0:
        print(f"  {'reg_w':>8} {'Feature set':<16} {'Split':<10} "
              f"{'Alive':>8} {'Total':>8} {'%Alive':>7} {'Test AUC':>9} {'Epoch':>6}")
        print("  " + "-" * 80)
        for _, row in binary_curve.sort_values(
            ["feature_set", "split", "reg_weight"]
        ).iterrows():
            print(f"  {row['reg_weight']:>8.3f} {row['feature_set']:<16} {row['split']:<10} "
                  f"{row['surviving_connections']:>8.0f} {row['total_connections']:>8.0f} "
                  f"{row['pct_alive']:>6.1f}% {row.get('test_auc', 0):>9.4f} "
                  f"{row['best_epoch']:>6.0f}")

    # ── Continuous Derived AUC vs surviving connections ──
    print("\n" + "=" * 70)
    print("  PRUNING CURVE — Continuous Derived AUC vs Surviving Connections")
    print("=" * 70 + "\n")

    cont_curve = curve_df[curve_df["target"] == "continuous"]
    if len(cont_curve) > 0:
        print(f"  {'reg_w':>8} {'Feature set':<16} {'Split':<10} "
              f"{'Alive':>8} {'%Alive':>7} {'Test R²':>9} {'Der. AUC':>9} {'Epoch':>6}")
        print("  " + "-" * 80)
        for _, row in cont_curve.sort_values(
            ["feature_set", "split", "reg_weight"]
        ).iterrows():
            print(f"  {row['reg_weight']:>8.3f} {row['feature_set']:<16} {row['split']:<10} "
                  f"{row['surviving_connections']:>8.0f} "
                  f"{row['pct_alive']:>6.1f}% {row.get('test_r2', 0):>9.4f} "
                  f"{row.get('test_derived_auc', 0):>9.4f} {row['best_epoch']:>6.0f}")

    # ── Mean across splits ──
    print("\n" + "=" * 70)
    print("  MEAN ACROSS SPLITS — Binary Test AUC")
    print("=" * 70 + "\n")

    if len(binary_curve) > 0:
        pivot = binary_curve.pivot_table(
            index=["feature_set", "reg_weight"],
            values=["test_auc", "surviving_connections", "pct_alive"],
            aggfunc="mean",
        )
        print(pivot.round(4).to_string())

    # ── Comparison point: Sparse KAN equivalent ──
    print("\n" + "=" * 70)
    print("  CLOSEST TO SPARSE KAN (~815 connections)")
    print("=" * 70 + "\n")

    sparse_kan_target = 815
    for fs in FEATURE_SETS:
        for tt in TARGET_TYPES:
            subset = curve_df[
                (curve_df["feature_set"] == fs) &
                (curve_df["target"] == tt)
            ].copy()
            if len(subset) == 0:
                continue

            mean_by_rw = subset.groupby("reg_weight")["surviving_connections"].mean()
            closest_rw = mean_by_rw.index[
                (mean_by_rw - sparse_kan_target).abs().argmin()
            ]
            closest_rows = subset[subset["reg_weight"] == closest_rw]

            metric_col = "test_auc" if tt == "binary" else "test_derived_auc"
            mean_metric = closest_rows[metric_col].mean() if metric_col in closest_rows else 0
            mean_conns = closest_rows["surviving_connections"].mean()

            print(f"  {fs:<16} {tt:<12} reg_weight={closest_rw:.3f}  "
                  f"connections={mean_conns:.0f}  "
                  f"{'AUC' if tt == 'binary' else 'Der.AUC'}={mean_metric:.4f}")

else:
    print("\n  No results to display.")

# %% [markdown]
# ## File Inventory

# %%
print("\n" + "=" * 70)
print("  SAVED FILES")
print("=" * 70)

for subdir in ["predictions", "metrics", "checkpoints", "optuna"]:
    d = RESULTS_DIR / subdir
    if d.exists():
        files = list(d.glob("*"))
        print(f"\n  {subdir}/: {len(files)} files")
        for f in sorted(files)[:3]:
            print(f"    {f.name}")
        if len(files) > 3:
            print(f"    ... and {len(files) - 3} more")

curve_path = RESULTS_DIR / "pruning_curve_mlp.csv"
if curve_path.exists():
    print(f"\n  pruning_curve_mlp.csv: {len(pd.read_csv(curve_path))} rows")

# %%

c:\Users\Henry\anaconda3\envs\diss\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: CPU
  DENSE MLP PRUNING SWEEP
  Architecture: [n_feat, 99, 13, 1] + SiLU
  Parameters: full_moments=219,609, means_only=72,099
  reg_weights: [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
  Optuna: 20 trials per run (lr, weight_decay, batch_size)
  Total runs: 128
  Results saving to: ..\..\..\Data\Results\Dense_vs_Sparse_KAN\pruning_sweep_mlp

────────────────────────────────────────────────────────────
  full_moments / Split_A / binary / reg_weight=0.01
────────────────────────────────────────────────────────────


[I 2026-07-25 23:57:51,350] Using an existing study with name 'dense_mlp_pruned_full_moments_binary_Split_A_rw0p0100' instead of creating a new one.


  Optuna: 20 trials already complete — skipping
  Best AUC: 0.7705  params: {'lr': 0.002345358480353672, 'weight_decay': 9.903880469449406e-06, 'batch_size': 64}
  Epoch    1 | Train loss 0.9898 | Val loss 1.1099  AUC 0.7362 | LR 2.3e-03
  Epoch   20 | Train loss 0.5314 | Val loss 1.2745  AUC 0.6687 | LR 5.9e-04
  Early stop at epoch 23. Best val AUC: 0.7507 at epoch 3
  Training complete in 6.4s

  Surviving connections: 538 / 219,496 (0.2%)
    Layer 0 (2204→99): 501 / 218,196 (0.2%)
    Layer 1 (99→13): 29 / 1,287 (2.3%)
    Layer 2 (13→1): 8 / 13 (61.5%)
  Test AUC: 0.7791  Train AUC: 0.8844  Gap: +0.1053

  ✓ 1/128 complete  (0min elapsed, ~22min remaining)

────────────────────────────────────────────────────────────
  full_moments / Split_A / binary / reg_weight=0.05
────────────────────────────────────────────────────────────
  Optuna: 20 trials already complete — skipping
  Best AUC: 0.7729  params: {'lr': 0.00014250573682600816, 'weight_decay': 0.0033098578498904057, 'batch_s

Best trial: 1. Best value: 9.6829e-05: 100%|██████████| 11/11 [01:36<00:00,  8.75s/it]


...

  Best MSE: 0.0001  params: {'lr': 0.004366664735068759, 'weight_decay': 4.62120670306908e-05, 'batch_size': 64}
  Epoch    1 | Train loss 0.020233 | Val MSE 0.003375  R² -28.9886 | LR 4.4e-03
  Epoch   20 | Train loss 0.000172 | Val MSE 0.000117  R² -0.0524 | LR 1.1e-03
  Early stop at epoch 22. Best val MSE: 0.0001 at epoch 2
  Training complete in 6.0s

  Surviving connections: 0 / 71,986 (0.0%)
    Layer 0 (714→99): 0 / 70,686 (0.0%)
    Layer 1 (99→13): 0 / 1,287 (0.0%)
    Layer 2 (13→1): 0 / 13 (0.0%)
  Test R²: -0.2273  Derived AUC: 0.5

  ✓ 126/128 complete  (173min elapsed, ~3min remaining)

────────────────────────────────────────────────────────────
  means_only / Split_D / continuous / reg_weight=0.7
────────────────────────────────────────────────────────────
  Optuna: 20 trials (0 already complete)


Best trial: 13. Best value: 0.000113138: 100%|██████████| 20/20 [02:34<00:00,  7.72s/it]


  Best MSE: 0.0001  params: {'lr': 0.00043842653432684623, 'weight_decay': 5.578992030971418e-06, 'batch_size': 64}
  Epoch    1 | Train loss 0.001782 | Val MSE 0.000810  R² -6.2354 | LR 4.4e-04
  Epoch   20 | Train loss 0.000171 | Val MSE 0.000117  R² -0.0569 | LR 1.1e-04
  Early stop at epoch 22. Best val MSE: 0.0001 at epoch 2
  Training complete in 5.9s

  Surviving connections: 580 / 71,986 (0.8%)
    Layer 0 (714→99): 0 / 70,686 (0.0%)
    Layer 1 (99→13): 568 / 1,287 (44.1%)
    Layer 2 (13→1): 12 / 13 (92.3%)
  Test R²: -0.0121  Derived AUC: 0.5

  ✓ 127/128 complete  (176min elapsed, ~1min remaining)

────────────────────────────────────────────────────────────
  means_only / Split_D / continuous / reg_weight=1.0
────────────────────────────────────────────────────────────
  Optuna: 20 trials (0 already complete)


Best trial: 8. Best value: 0.00011304: 100%|██████████| 20/20 [02:50<00:00,  8.51s/it]


  Best MSE: 0.0001  params: {'lr': 0.00035958850306931346, 'weight_decay': 3.5806075830615845e-06, 'batch_size': 64}
  Epoch    1 | Train loss 0.023926 | Val MSE 0.016143  R² -142.4951 | LR 3.6e-04
  Epoch   20 | Train loss 0.000186 | Val MSE 0.000143  R² -0.2818 | LR 3.6e-04
  Early stop at epoch 38. Best val MSE: 0.0002 at epoch 18
  Training complete in 10.0s

  Surviving connections: 0 / 71,986 (0.0%)
    Layer 0 (714→99): 0 / 70,686 (0.0%)
    Layer 1 (99→13): 0 / 1,287 (0.0%)
    Layer 2 (13→1): 0 / 13 (0.0%)
  Test R²: -0.2631  Derived AUC: 0.5

  ✓ 128/128 complete  (179min elapsed, ~0min remaining)


  FINISHED: 128/128 completed, 0 failed
  Total time: 178.9 minutes (3.0 hours)

  PRUNING CURVE — Binary Test AUC vs Surviving Connections

     reg_w Feature set      Split         Alive    Total  %Alive  Test AUC  Epoch
  --------------------------------------------------------------------------------
     0.010 full_moments     Split_A         538   219496    0.2%    0.7791   

# with Dropout

In [2]:
# %% [markdown]
# # 05c — Dense MLP with Input Dropout
#
# Mirrors 02c (Dense KAN + dropout) for the MLP.
# Tests whether input dropout fixes the continuous AUC=0.5 problem
# for the MLP, and whether the fix produces different results than
# the KAN version (isolating splines vs SiLU under dropout).
#
# Architecture: [Dropout, Linear, SiLU, Linear, SiLU, Linear]
# Same layer widths as KAN: [n_features, 99, 13, 1]
# No L1 regularisation — dropout replaces it.
# Weight decay [1e-4, 1e-1], batch size [256, 512, 1024].
#
# 4 splits × 2 feature sets × 2 targets = 16 runs, 40 Optuna trials each.
# Estimated runtime: ~2-3 hours on local CPU.

# %%
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import time
import torch
import torch.nn as nn
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device
from training import train_model, save_checkpoint
from evaluation import evaluate_model, save_predictions, compute_calibration

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

SPLITS_DIR = Path("../../..") / "Data" / "Splits"
RESULTS_DIR = Path("../../..") / "Data" / "Results" / "Dense_vs_Sparse_KAN" / "dropout_mlp"

FEATURE_SETS = ["full_moments", "means_only"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS = ["Split_A", "Split_B", "Split_C", "Split_D"]

# ── Fixed architecture (matches KAN layer widths) ──
N_SUBTHEMES = 99
N_THEMES = 13

# ── Optuna settings ──
N_TRIALS = 40


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL: MLP WITH INPUT DROPOUT
# ═══════════════════════════════════════════════════════════════════════════════

def make_mlp_with_dropout(n_features, dropout_rate):
    """
    Dense MLP with input dropout as first layer.
    SiLU activation to match KAN's base component.
    Dropout zeroes features before the network sees them —
    same mechanism as the KAN dropout wrapper.
    """
    return nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(n_features, N_SUBTHEMES),
        nn.SiLU(),
        nn.Linear(N_SUBTHEMES, N_THEMES),
        nn.SiLU(),
        nn.Linear(N_THEMES, 1),
    )


def make_mlp_no_dropout(n_features):
    """MLP without dropout for the full batch sanity check."""
    return nn.Sequential(
        nn.Linear(n_features, N_SUBTHEMES),
        nn.SiLU(),
        nn.Linear(N_SUBTHEMES, N_THEMES),
        nn.SiLU(),
        nn.Linear(N_THEMES, 1),
    )


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL FACTORY FOR OPTUNA
# ═══════════════════════════════════════════════════════════════════════════════

def make_model_factory(n_features):
    """
    Factory for Optuna. Same 4 hyperparameters as KAN dropout:
        lr:            [1e-4, 1e-2]       log
        weight_decay:  [1e-4, 1e-1]       log — wider/higher for dropout scaling
        batch_size:    [256, 512, 1024]    categorical — larger for cleaner gradients
        dropout_rate:  [0.7, 0.8, 0.9]    categorical

    No L1 — dropout provides regularisation.
    """
    def factory(trial):
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
        dropout_rate = trial.suggest_categorical("dropout_rate", [0.7, 0.8, 0.9])

        model = make_mlp_with_dropout(n_features, dropout_rate)

        train_kwargs = {
            "lr": lr,
            "weight_decay": weight_decay,
            "n_epochs": 300,
            "patience": 20,
        }

        return model, train_kwargs

    return factory


# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE EXPERIMENT
# ═══════════════════════════════════════════════════════════════════════════════

def run_single_experiment(split_name, feature_set, target_type, device):
    """Run Optuna → train → evaluate → save for one configuration."""
    model_name = f"dense_mlp_dropout_{feature_set}"

    print(f"\n{'─'*60}")
    print(f"  {feature_set} / {split_name} / {target_type}")
    print(f"{'─'*60}")

    data = load_split(split_name, feature_set, SPLITS_DIR)
    n_features = data["n_features"]

    n_pos = data["y_train"].sum()
    n_neg = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    # ── OPTUNA SEARCH ──
    factory = make_model_factory(n_features)
    direction = "maximize" if target_type == "binary" else "minimize"

    study_name = f"{model_name}_{target_type}_{split_name}"
    study_path = RESULTS_DIR / "optuna" / f"{study_name}.db"
    study_path.parent.mkdir(parents=True, exist_ok=True)
    storage = f"sqlite:///{study_path}"

    study = optuna.create_study(
        study_name=study_name,
        storage=storage,
        direction=direction,
        load_if_exists=True,
    )

    def objective(trial):
        model, train_kwargs = factory(trial)
        batch_size = trial.params["batch_size"]

        loaders = get_dataloaders(
            split_name, feature_set, SPLITS_DIR,
            target_type=target_type,
            batch_size=batch_size,
        )

        result = train_model(
            model=model,
            train_loader=loaders["train"],
            val_loader=loaders["val"],
            device=device,
            target_type=target_type,
            pos_weight=pos_weight if target_type == "binary" else None,
            verbose=False,
            **train_kwargs,
        )

        return result["best_val_metric"]

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    existing = len([t for t in study.trials
                    if t.state == optuna.trial.TrialState.COMPLETE])
    remaining = max(0, N_TRIALS - existing)

    if remaining > 0:
        print(f"  Optuna: {remaining} trials ({existing} already complete)")
        study.optimize(objective, n_trials=remaining, show_progress_bar=True)
    else:
        print(f"  Optuna: {existing} trials already complete — skipping")

    best_params = study.best_params
    metric_name = "AUC" if target_type == "binary" else "MSE"
    print(f"  Best {metric_name}: {study.best_value:.4f}")
    print(f"  Best params: {best_params}")

    # ── FINAL TRAINING ──
    batch_size = best_params.get("batch_size", 512)
    dropout_rate = best_params.get("dropout_rate", 0.8)

    loaders = get_dataloaders(
        split_name, feature_set, SPLITS_DIR,
        target_type=target_type,
        batch_size=batch_size,
    )

    model = make_mlp_with_dropout(n_features, dropout_rate)

    result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        target_type=target_type,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        pos_weight=pos_weight if target_type == "binary" else None,
        n_epochs=300,
        patience=20,
        verbose=True,
        log_every=20,
    )

    # ── EVALUATE ──
    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        if target_type == "binary":
            cal = compute_calibration(metrics["y_true"], metrics["y_prob"])
            metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            dates=data[f"dates_{part}"],
            returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters=best_params if part == "test" else None,
            results_dir=RESULTS_DIR,
        )

        all_metrics[part] = metrics

    # ── SAVE CHECKPOINT ──
    ckpt_dir = RESULTS_DIR / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model,
        train_result=result,
        hyperparameters=best_params,
        model_config={
            "layers": [n_features, N_SUBTHEMES, N_THEMES, 1],
            "activation": "SiLU",
            "dropout_rate": dropout_rate,
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}.pt",
    )

    # ── PRINT SUMMARY ──
    if target_type == "binary":
        print(f"\n  Results:")
        print(f"    Train AUC: {all_metrics['train']['auc']:.4f}")
        print(f"    Val AUC:   {all_metrics['val']['auc']:.4f}")
        print(f"    Test AUC:  {all_metrics['test']['auc']:.4f}")
        print(f"    Gap:       {all_metrics['train']['auc'] - all_metrics['test']['auc']:+.4f}")
    else:
        print(f"\n  Results:")
        print(f"    Train R²:       {all_metrics['train']['r2']:.4f}")
        print(f"    Val R²:         {all_metrics['val']['r2']:.4f}")
        print(f"    Test R²:        {all_metrics['test']['r2']:.4f}")
        print(f"    Derived AUC:    {all_metrics['test']['derived_auc']:.4f}")

    return {
        "best_params": best_params,
        "metrics": all_metrics,
        "best_epoch": result["best_epoch"],
        "total_time": result["total_time"],
    }


# %% [markdown]
# ## Full Batch Sanity Check
# Same as 02c: one quick test — does full batch alone fix continuous AUC=0.5?

# %%
device = get_device()

print("=" * 70)
print("  SANITY CHECK: Full batch MLP, no dropout, continuous Split A")
print("=" * 70)

data = load_split("Split_A", "full_moments", SPLITS_DIR)
n_features = data["n_features"]

loaders = get_dataloaders(
    "Split_A", "full_moments", SPLITS_DIR,
    target_type="continuous",
    batch_size=len(data["X_train"]),
)

model_sanity = make_mlp_no_dropout(n_features)

result_sanity = train_model(
    model=model_sanity,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    device=device,
    target_type="continuous",
    lr=1e-3,
    weight_decay=1e-4,
    n_epochs=300,
    patience=20,
    verbose=True,
    log_every=10,
)

metrics_sanity = evaluate_model(
    model_sanity, loaders["test"], device, "continuous",
    y_true_binary=data["y_test"],
)

print(f"\n  Full batch result:")
print(f"    Test R²:        {metrics_sanity['r2']:.4f}")
print(f"    Derived AUC:    {metrics_sanity['derived_auc']:.4f}")
print(f"    Best epoch:     {result_sanity['best_epoch']}")

if metrics_sanity["derived_auc"] > 0.52:
    print(f"\n  ✓ Full batch broke through 0.5!")
    print(f"  Full batch alone may be sufficient — review before proceeding.")
    print(f"  Set SKIP_DROPOUT = False below to run dropout experiments anyway.")
    SKIP_DROPOUT = True
else:
    print(f"\n  ✗ Still at 0.5. Proceeding with dropout experiment.")
    SKIP_DROPOUT = False

del model_sanity

# %% [markdown]
# ## Run All Dropout Experiments

# %%
total_runs = len(FEATURE_SETS) * len(TARGET_TYPES) * len(ALL_SPLITS)

n_params_full = sum(p.numel() for p in make_mlp_no_dropout(2204).parameters())
n_params_means = sum(p.numel() for p in make_mlp_no_dropout(714).parameters())

print("\n" + "=" * 70)
print("  DENSE MLP + DROPOUT: 4 Splits × 2 Feature Sets × 2 Targets")
print(f"  Architecture: [n_feat, {N_SUBTHEMES}, {N_THEMES}, 1] + SiLU + input dropout")
print(f"  Parameters: full_moments={n_params_full:,}, means_only={n_params_means:,}")
print(f"  Optuna: {N_TRIALS} trials per run")
print(f"  Search: lr, weight_decay [1e-4,1e-1], batch [256,512,1024], dropout [0.7,0.8,0.9]")
print(f"  No L1 — dropout provides regularisation")
print(f"  Total runs: {total_runs}")
print(f"  Results: {RESULTS_DIR}")
print("=" * 70)

all_results = []
best_params_store = {}
completed = 0
failed = 0

if SKIP_DROPOUT:
    print("\n  SKIP_DROPOUT = True — skipping dropout experiments.")
    print("  Full batch sanity check succeeded. Review results above.")
    print("  Set SKIP_DROPOUT = False and rerun this cell to proceed anyway.")
else:
    total_start = time.time()

    for feature_set in FEATURE_SETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                try:
                    exp = run_single_experiment(
                        split_name, feature_set, target_type, device,
                    )

                    all_results.append({
                        "feature_set": feature_set,
                        "split": split_name,
                        "target": target_type,
                        **{f"test_{k}": v for k, v in exp["metrics"]["test"].items()
                           if not isinstance(v, np.ndarray)},
                        "best_epoch": exp["best_epoch"],
                        "time_s": exp["total_time"],
                        "dropout_rate": exp["best_params"].get("dropout_rate"),
                    })

                    best_params_store[(feature_set, target_type, split_name)] = exp["best_params"]
                    completed += 1

                    elapsed = time.time() - total_start
                    rate = elapsed / completed
                    remaining_est = rate * (total_runs - completed)
                    print(f"\n  ✓ {completed}/{total_runs} complete  "
                          f"({elapsed/60:.0f}min elapsed, ~{remaining_est/60:.0f}min remaining)")

                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): {feature_set}/{split_name}/{target_type}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

    total_time = time.time() - total_start
    print(f"\n\n{'='*70}")
    print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
    print(f"  Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
    print(f"{'='*70}")

# %% [markdown]
# ## Results Summary

# %%
if all_results:
    results_df = pd.DataFrame(all_results)

    # ── Binary AUC ──
    print("\n" + "=" * 70)
    print("  DENSE MLP + DROPOUT — Binary Test AUC")
    print("=" * 70 + "\n")

    binary_df = results_df[results_df["target"] == "binary"]
    if "test_auc" in binary_df.columns and len(binary_df) > 0:
        pivot = binary_df.pivot_table(index="feature_set", columns="split", values="test_auc")
        pivot["Mean"] = pivot.mean(axis=1)
        print(pivot.round(4).to_string())

    # ── Continuous R² ──
    print("\n" + "=" * 70)
    print("  DENSE MLP + DROPOUT — Continuous Test R²")
    print("=" * 70 + "\n")

    cont_df = results_df[results_df["target"] == "continuous"]
    if "test_r2" in cont_df.columns and len(cont_df) > 0:
        pivot = cont_df.pivot_table(index="feature_set", columns="split", values="test_r2")
        pivot["Mean"] = pivot.mean(axis=1)
        print(pivot.round(4).to_string())

    # ── THE KEY METRIC: Continuous Derived AUC ──
    print("\n" + "=" * 70)
    print("  DENSE MLP + DROPOUT — Continuous Derived AUC")
    print("  (Was 0.5 without dropout — did it break through?)")
    print("=" * 70 + "\n")

    if "test_derived_auc" in cont_df.columns and len(cont_df) > 0:
        pivot = cont_df.pivot_table(index="feature_set", columns="split", values="test_derived_auc")
        pivot["Mean"] = pivot.mean(axis=1)
        print(pivot.round(4).to_string())

    # ── Best hyperparameters ──
    print("\n" + "=" * 70)
    print("  BEST HYPERPARAMETERS")
    print("=" * 70 + "\n")

    print(f"  {'Feature set':<16} {'Target':<12} {'Split':<10} "
          f"{'lr':>10} {'wd':>10} {'bs':>5} {'drop':>5}")
    print("  " + "-" * 70)
    for (fs, tt, split), params in sorted(best_params_store.items()):
        print(f"  {fs:<16} {tt:<12} {split:<10} "
              f"{params['lr']:>10.6f} {params['weight_decay']:>10.6f} "
              f"{params['batch_size']:>5} {params['dropout_rate']:>5.1f}")

    # ── Preferred dropout rates ──
    print("\n" + "=" * 70)
    print("  DROPOUT RATE DISTRIBUTION")
    print("=" * 70 + "\n")

    if "dropout_rate" in results_df.columns:
        for tt in TARGET_TYPES:
            subset = results_df[results_df["target"] == tt]
            if len(subset) > 0:
                counts = subset["dropout_rate"].value_counts().sort_index()
                print(f"  {tt}:")
                for rate, count in counts.items():
                    print(f"    dropout={rate}: {count}/{len(subset)} runs")
                print()

    # ── Timing ──
    print("\n" + "=" * 70)
    print("  TIMING")
    print("=" * 70 + "\n")

    for _, row in results_df.iterrows():
        print(f"  {row['feature_set']:<16} {row['split']:<10} {row['target']:<12} "
              f"epoch {row['best_epoch']:>3}  {row['time_s']:>6.1f}s")

    # ── Comparison with previous Dense MLP (no dropout) ──
    print("\n" + "=" * 70)
    print("  COMPARISON: Dense MLP without dropout (from notebook 05)")
    print("=" * 70 + "\n")

    print("  Without dropout (continuous): derived AUC = 0.5 everywhere")
    print("  Without dropout (binary):     mean AUC ≈ 0.68")
    print()
    if "test_derived_auc" in cont_df.columns and len(cont_df) > 0:
        mean_dauc = cont_df["test_derived_auc"].mean()
        print(f"  With dropout (continuous):    mean derived AUC = {mean_dauc:.4f}")
    if "test_auc" in binary_df.columns and len(binary_df) > 0:
        mean_auc = binary_df["test_auc"].mean()
        print(f"  With dropout (binary):        mean AUC = {mean_auc:.4f}")

else:
    print("  No results to display.")

# %% [markdown]
# ## File Inventory

# %%
print("\n" + "=" * 70)
print("  SAVED FILES")
print("=" * 70)

for subdir in ["predictions", "metrics", "checkpoints", "optuna"]:
    d = RESULTS_DIR / subdir
    if d.exists():
        files = list(d.glob("*"))
        print(f"\n  {subdir}/: {len(files)} files")
        for f in sorted(files)[:3]:
            print(f"    {f.name}")
        if len(files) > 3:
            print(f"    ... and {len(files) - 3} more")

# %%

Device: CPU
  SANITY CHECK: Full batch MLP, no dropout, continuous Split A
  Epoch    1 | Train loss 0.011603 | Val MSE 0.087035  R² -307.2279 | LR 1.0e-03
  Epoch   10 | Train loss 0.001108 | Val MSE 0.004411  R² -14.6223 | LR 1.0e-03
  Epoch   20 | Train loss 0.000745 | Val MSE 0.001176  R² -3.1655 | LR 5.0e-04
  Epoch   30 | Train loss 0.000351 | Val MSE 0.003174  R² -10.2420 | LR 2.5e-04
  Early stop at epoch 36. Best val MSE: 0.0012 at epoch 16
  Training complete in 4.5s

  Full batch result:
    Test R²:        -71.8760
    Derived AUC:    0.5000
    Best epoch:     16

  ✗ Still at 0.5. Proceeding with dropout experiment.

  DENSE MLP + DROPOUT: 4 Splits × 2 Feature Sets × 2 Targets
  Architecture: [n_feat, 99, 13, 1] + SiLU + input dropout
  Parameters: full_moments=219,609, means_only=72,099
  Optuna: 40 trials per run
  Search: lr, weight_decay [1e-4,1e-1], batch [256,512,1024], dropout [0.7,0.8,0.9]
  No L1 — dropout provides regularisation
  Total runs: 16
  Results: ..\..

Best trial: 32. Best value: 0.768428: 100%|██████████| 40/40 [04:11<00:00,  6.30s/it]


  Best AUC: 0.7684
  Best params: {'lr': 0.007015493758462789, 'weight_decay': 0.04937044032871191, 'batch_size': 1024, 'dropout_rate': 0.8}
  Epoch    1 | Train loss 1.0487 | Val loss 1.0822  AUC 0.7065 | LR 7.0e-03
  Epoch   20 | Train loss 0.4491 | Val loss 1.2089  AUC 0.6508 | LR 1.8e-03
  Early stop at epoch 25. Best val AUC: 0.7407 at epoch 5
  Training complete in 5.2s

  Results:
    Train AUC: 0.9225
    Val AUC:   0.7407
    Test AUC:  0.7166
    Gap:       +0.2059

  ✓ 1/16 complete  (4min elapsed, ~65min remaining)

────────────────────────────────────────────────────────────
  full_moments / Split_B / binary
────────────────────────────────────────────────────────────
  Optuna: 40 trials (0 already complete)


Best trial: 26. Best value: 0.831667: 100%|██████████| 40/40 [03:54<00:00,  5.85s/it]


  Best AUC: 0.8317
  Best params: {'lr': 0.006317514981211228, 'weight_decay': 0.017343161889578147, 'batch_size': 512, 'dropout_rate': 0.9}
  Epoch    1 | Train loss 0.9821 | Val loss 0.5076  AUC 0.5553 | LR 6.3e-03
  Epoch   20 | Train loss 0.6138 | Val loss 0.4112  AUC 0.7501 | LR 3.2e-03
  Early stop at epoch 33. Best val AUC: 0.7800 at epoch 13
  Training complete in 4.8s

  Results:
    Train AUC: 0.9251
    Val AUC:   0.7800
    Test AUC:  0.7234
    Gap:       +0.2017

  ✓ 2/16 complete  (8min elapsed, ~58min remaining)

────────────────────────────────────────────────────────────
  full_moments / Split_C / binary
────────────────────────────────────────────────────────────
  Optuna: 40 trials (0 already complete)


...

  Best MSE: 0.0000
  Best params: {'lr': 0.004506270649347052, 'weight_decay': 0.002191625523630044, 'batch_size': 512, 'dropout_rate': 0.8}
  Epoch    1 | Train loss 0.116330 | Val MSE 0.034292  R² -961.4344 | LR 4.5e-03
  Epoch   20 | Train loss 0.000172 | Val MSE 0.000071  R² -0.9818 | LR 2.3e-03
  Early stop at epoch 25. Best val MSE: 0.0002 at epoch 5
  Training complete in 1.9s

  Results:
    Train R²:       -2.9668
    Val R²:         -3.2582
    Test R²:        -1.6421
    Derived AUC:    0.5000

  ✓ 14/16 complete  (57min elapsed, ~8min remaining)

────────────────────────────────────────────────────────────
  means_only / Split_C / continuous
────────────────────────────────────────────────────────────
  Optuna: 40 trials (0 already complete)


Best trial: 16. Best value: 0.000118252: 100%|██████████| 40/40 [02:49<00:00,  4.24s/it]


  Best MSE: 0.0001
  Best params: {'lr': 0.0020114980924823716, 'weight_decay': 0.0005889943580527739, 'batch_size': 512, 'dropout_rate': 0.7}
  Epoch    1 | Train loss 0.026939 | Val MSE 0.011031  R² -114.1368 | LR 2.0e-03
  Epoch   20 | Train loss 0.000248 | Val MSE 0.000280  R² -1.9246 | LR 2.0e-03
  Early stop at epoch 33. Best val MSE: 0.0003 at epoch 13
  Training complete in 3.0s

  Results:
    Train R²:       -0.0218
    Val R²:         -1.9645
    Test R²:        -0.9365
    Derived AUC:    0.5000

  ✓ 15/16 complete  (60min elapsed, ~4min remaining)

────────────────────────────────────────────────────────────
  means_only / Split_D / continuous
────────────────────────────────────────────────────────────
  Optuna: 40 trials (0 already complete)


Best trial: 13. Best value: 0.000101401: 100%|██████████| 40/40 [03:37<00:00,  5.43s/it]


  Best MSE: 0.0001
  Best params: {'lr': 0.002095582833420121, 'weight_decay': 0.001981273536095156, 'batch_size': 1024, 'dropout_rate': 0.7}
  Epoch    1 | Train loss 0.031945 | Val MSE 0.004284  R² -37.1211 | LR 2.1e-03
  Epoch   20 | Train loss 0.000197 | Val MSE 0.001075  R² -8.5700 | LR 2.1e-03
  Early stop at epoch 34. Best val MSE: 0.0011 at epoch 14
  Training complete in 3.4s

  Results:
    Train R²:       0.2584
    Val R²:         -8.5429
    Test R²:        -73.9154
    Derived AUC:    0.5000

  ✓ 16/16 complete  (64min elapsed, ~0min remaining)


  FINISHED: 16/16 completed, 0 failed
  Total time: 64.0 minutes (1.1 hours)

  DENSE MLP + DROPOUT — Binary Test AUC

split         Split_A  Split_B  Split_C  Split_D    Mean
feature_set                                             
full_moments   0.7166   0.7234   0.7164   0.6497  0.7015
means_only     0.7583   0.7013   0.7064   0.6170  0.6958

  DENSE MLP + DROPOUT — Continuous Test R²

split         Split_A  Split_B  Split_C  

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# Load one set of predictions
pred_path = "../../../Data/Results/Dense_vs_Sparse_KAN/dropout_mlp/predictions"
df = pd.read_parquet(f"{pred_path}/dense_mlp_dropout_full_moments_continuous_Split_A_test.parquet")

print("Prediction statistics:")
print(f"  y_pred mean: {df['y_pred'].mean():.8f}")
print(f"  y_pred std:  {df['y_pred'].std():.8f}")
print(f"  y_pred min:  {df['y_pred'].min():.8f}")
print(f"  y_pred max:  {df['y_pred'].max():.8f}")
print(f"  y_pred range: {df['y_pred'].max() - df['y_pred'].min():.8f}")
print()

# Check if predictions are essentially constant
print(f"  Unique values (first 10): {sorted(df['y_pred'].unique())[:10]}")
print()

# Manually compute AUC both ways
y_true_binary = (df['y_true'] < -0.02).astype(int)
print(f"  Crash days: {y_true_binary.sum()}/{len(y_true_binary)}")
print()

# AUC with predictions as-is (more negative = higher risk)
auc_raw = roc_auc_score(y_true_binary, -df['y_pred'])
print(f"  AUC (negated predictions): {auc_raw:.4f}")

# AUC with predictions not negated
auc_pos = roc_auc_score(y_true_binary, df['y_pred'])
print(f"  AUC (raw predictions):     {auc_pos:.4f}")

Prediction statistics:
  y_pred mean: 0.00839550
  y_pred std:  0.01617719
  y_pred min:  -0.02306600
  y_pred max:  0.05472791
  y_pred range: 0.07779391

  Unique values (first 10): [np.float32(-0.023066), np.float32(-0.022158027), np.float32(-0.021246292), np.float32(-0.021088451), np.float32(-0.020696796), np.float32(-0.020695142), np.float32(-0.020564944), np.float32(-0.020562366), np.float32(-0.019952767), np.float32(-0.01980193)]



KeyError: 'y_true'

In [8]:
from sklearn.metrics import roc_auc_score
from data_utils import load_split
from pathlib import Path
import pandas as pd

data = load_split("Split_A", "full_moments", Path("../../..") / "Data" / "Splits")
df = pd.read_parquet(
    "../../../Data/Results/Dense_vs_Sparse_KAN/dropout_mlp/predictions/"
    "dense_mlp_dropout_full_moments_continuous_Split_A_test.parquet"
)

y_pred = df["y_pred"].values
y_binary = data["y_test"]          # actual binary labels
y_minret = data["minret_test"]     # actual continuous minret

print(f"y_binary unique: {sorted(set(y_binary))}")
print(f"y_minret range:  [{y_minret.min():.4f}, {y_minret.max():.4f}]")
print(f"y_pred range:    [{y_pred.min():.4f}, {y_pred.max():.4f}]")
print()

# The CORRECT derived AUC: negate predictions, score against actual binary labels
auc_correct = roc_auc_score(y_binary, -y_pred)
print(f"CORRECT derived AUC: {auc_correct:.4f}")

y_binary unique: [np.float32(0.0), np.float32(1.0)]
y_minret range:  [-0.0463, 0.0050]
y_pred range:    [-0.0231, 0.0547]

CORRECT derived AUC: 0.8161
